Nettoyage et vérifications de base sur le dataset symptômes -> maladies.

Ce script tourne AVANT train_model.py. Il produit un CSV nettoyé dans
data/processed/, que train_model.py peut ensuite charger directement
(plus besoin de refiltrer les classes rares à l'entraînement).

Vérifications effectuées :
1. Valeurs manquantes (NaN)
2. Lignes en double
3. Colonnes de symptômes non strictement binaires (0/1)
4. Symptômes qui n'apparaissent jamais (colonne à 0 partout -> aucune info)
5. Maladies avec trop peu d'exemples (configurable, MIN_SAMPLES_PER_CLASS)

In [2]:
import os
import pandas as pd

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
SRC = os.path.join(SCRIPT_DIR, "..", "..", "data", "raw",
                    "Final_Augmented_dataset_Diseases_and_Symptoms.csv")
OUT = os.path.join(SCRIPT_DIR, "..", "..", "data", "cleaned_dataset.csv")

TARGET_COLUMN = "diseases"

# Toute maladie avec moins de ce nombre d'exemples est retirée. 2 est le
# minimum strict pour pouvoir faire un split train/test ; on part sur 2 ici
# (on retire donc les singletons demandés), mais train_model.py applique
# ensuite son propre seuil (5) qui peut être plus strict si besoin.
MIN_SAMPLES_PER_CLASS = 2

In [3]:
if not os.path.isfile(SRC):
    raise FileNotFoundError(f"Fichier introuvable : {os.path.abspath(SRC)}")
os.makedirs(os.path.dirname(OUT), exist_ok=True)

print("Chargement des données...")
df = pd.read_csv(SRC)
symptom_columns = [c for c in df.columns if c != TARGET_COLUMN]
print(f"Départ : {len(df)} lignes, {len(symptom_columns)} colonnes de symptômes, "
        f"{df[TARGET_COLUMN].nunique()} maladies")

Chargement des données...
Départ : 246945 lignes, 377 colonnes de symptômes, 773 maladies


In [4]:
# ------------------------------------------------------------------
# 1. Valeurs manquantes
# ------------------------------------------------------------------
n_missing = df.isna().sum().sum()
if n_missing > 0:
    rows_with_na = df[df.isna().any(axis=1)]
    print(f"[valeurs manquantes] {n_missing} valeurs NaN sur "
            f"{len(rows_with_na)} lignes -> ces lignes sont supprimées")
    df = df.dropna()
else:
    print("[valeurs manquantes] aucune")

[valeurs manquantes] aucune


In [5]:
# ------------------------------------------------------------------
# 2. Ligne cible (diseases) vide ou aberrante
# ------------------------------------------------------------------
n_empty_target = (df[TARGET_COLUMN].astype(str).str.strip() == "").sum()
if n_empty_target > 0:
    print(f"[cible vide] {n_empty_target} lignes sans nom de maladie -> supprimées")
    df = df[df[TARGET_COLUMN].astype(str).str.strip() != ""]
else:
    print("[cible vide] aucune")

[cible vide] aucune


In [6]:
# ------------------------------------------------------------------
# 3. Lignes strictement dupliquées
# ------------------------------------------------------------------
n_duplicates = df.duplicated().sum()
if n_duplicates > 0:
    print(f"[doublons] {n_duplicates} lignes strictement identiques trouvées "
            f"(gardées : ce dataset est volontairement augmenté, les doublons "
            f"exacts entre patients différents sont normaux ici)")
else:
    print("[doublons] aucun")

[doublons] 57298 lignes strictement identiques trouvées (gardées : ce dataset est volontairement augmenté, les doublons exacts entre patients différents sont normaux ici)


In [7]:
# ------------------------------------------------------------------
# 4. Colonnes de symptômes non binaires (autre chose que 0/1)
# ------------------------------------------------------------------
non_binary_cols = [
    c for c in symptom_columns
    if not df[c].dropna().isin([0, 1]).all()
]
if non_binary_cols:
    print(f"[colonnes non binaires] {len(non_binary_cols)} colonnes contiennent "
            f"des valeurs hors {{0, 1}} : {non_binary_cols[:10]}"
            f"{' ...' if len(non_binary_cols) > 10 else ''}")
    print("  -> à vérifier manuellement avant d'entraîner, ce script ne les corrige pas")
else:
    print("[colonnes non binaires] aucune, tous les symptômes sont bien en 0/1")

[colonnes non binaires] aucune, tous les symptômes sont bien en 0/1


In [11]:
# ------------------------------------------------------------------
# 5. Symptômes qui n'apparaissent jamais (colonne toujours à 0)
# ------------------------------------------------------------------
zero_variance_cols = [c for c in symptom_columns if df[c].nunique() <= 1]
if zero_variance_cols:
    print(f"[symptômes inutiles] {len(zero_variance_cols)} colonnes n'apportent "
            f"aucune information (valeur constante) : {zero_variance_cols[:10]}"
            f"{' ...' if len(zero_variance_cols) > 10 else ''}")
    print("  -> supprimées : le Random Forest les ignorait déjà, "
            "autant alléger le dataset")
    df = df.drop(columns=zero_variance_cols)
    symptom_columns = [c for c in symptom_columns if c not in zero_variance_cols]
else:
    print("[symptômes inutiles] aucune colonne constante")

[symptômes inutiles] 52 colonnes n'apportent aucune information (valeur constante) : ['pus in sputum', 'underweight', 'arm cramps or spasms', 'abnormal appearing tongue', 'pallor', 'shoulder cramps or spasms', 'joint stiffness or tightness', 'eye strain', 'pus in urine', 'abnormal size or shape of ear'] ...
  -> supprimées : le Random Forest les ignorait déjà, autant alléger le dataset


In [12]:
# ------------------------------------------------------------------
# 6. Maladies avec trop peu d'exemples
# ------------------------------------------------------------------
counts = df[TARGET_COLUMN].value_counts()
rare_diseases = counts[counts < MIN_SAMPLES_PER_CLASS].index.tolist()
if rare_diseases:
    print(f"[maladies rares] {len(rare_diseases)} maladies avec moins de "
            f"{MIN_SAMPLES_PER_CLASS} exemple(s) supprimées : "
            f"{rare_diseases[:10]}{' ...' if len(rare_diseases) > 10 else ''}")
    df = df[~df[TARGET_COLUMN].isin(rare_diseases)]
else:
    print("[maladies rares] aucune")

[maladies rares] aucune


In [13]:
# ------------------------------------------------------------------
# Résumé + sauvegarde
# ------------------------------------------------------------------
print(f"\nArrivée : {len(df)} lignes, {df[TARGET_COLUMN].nunique()} maladies conservées")
df.to_csv(OUT, index=False)
print(f"Dataset nettoyé sauvegardé : {os.path.abspath(OUT)}")


Arrivée : 246926 lignes, 754 maladies conservées
Dataset nettoyé sauvegardé : c:\ESGI\4annee\Trimestre_2\0_PA\MediGuess\data\cleaned_dataset.csv
